In [1]:
"""
PREAMBLE.
"""

# dependencies.
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from varclushi import VarClusHi
from sklearn.preprocessing import StandardScaler


# initialize logger.
log = logging.getLogger(__name__)
log.addHandler(logging.StreamHandler(stream=sys.stdout))
log.setLevel(logging.INFO)

log.info("Log initialized.")

# set seed.
RANDOM_SEED = 4890
np.random.seed(RANDOM_SEED)

# directories.
DATA_DIR = Path("../_data").resolve()
OUTPUT_DIR = Path("../output").resolve()

Log initialized.


In [2]:
"""
ENCODE FACTORIAL VARIABLES.
"""

# load training data.
df_train = pd.read_csv(DATA_DIR / "train.csv")

# create dummy variables for categorical/factor vars.
outcome_vars = ["clm", "numclaims", "claimcst0"]
pred_vars = list(set(df_train.columns).difference(outcome_vars + ["sample"]))
factor_vars = [v for v in pred_vars if df_train[v].dtype == "object"]

df_train = pd.get_dummies(
    data=df_train, 
    columns=factor_vars, 
    drop_first=False
)

# fix dummy cols w/ hyphen.
def rm_hypen(col: str) -> str:
    """Removes hypenations/spaces in cols and ensures lowercase."""
    
    col = col.strip()
    col = col.replace("-", "")
    col = col.replace(" ", "")
    col = col.lower()
    
    return col

df_train = df_train.rename(columns=rm_hypen)

# log the new dummy variables added
non_dummy = list(set(pred_vars).difference(factor_vars)) + outcome_vars + ["sample"]
dummy = list(set(df_train.columns).difference(set(non_dummy)))
all_cols = non_dummy + dummy

log.info(f"Dummy variables added: {sorted(dummy)}")


# clear unused mem.
del pred_vars, factor_vars, outcome_vars, non_dummy, dummy, all_cols

Dummy variables added: ['area_a', 'area_b', 'area_c', 'area_d', 'area_e', 'area_f', 'engine_type_dissel', 'engine_type_electric', 'engine_type_hybrid', 'engine_type_petrol', 'gender_f', 'gender_m', 'marital_status_m', 'marital_status_s', 'time_driven_12am6am', 'time_driven_12pm6pm', 'time_driven_6am12pm', 'time_driven_6pm12am', 'time_of_week_driven_weekday', 'time_of_week_driven_weekend', 'veh_body_bus', 'veh_body_convt', 'veh_body_coupe', 'veh_body_hdtop', 'veh_body_mcara', 'veh_body_mibus', 'veh_body_panvn', 'veh_body_rdstr', 'veh_body_sedan', 'veh_body_stnwg', 'veh_body_suv', 'veh_body_truck', 'veh_body_ute', 'veh_color_black', 'veh_color_blue', 'veh_color_brown', 'veh_color_gray', 'veh_color_green', 'veh_color_red', 'veh_color_silver', 'veh_color_white', 'veh_color_yellow']


In [3]:
"""
STANDARDIZE/PREPARE FEATURES.
"""

# split into the two samples.
df_build = df_train[df_train["sample"] == "1|bld"].copy()
df_valid = df_train[df_train["sample"] == "2|val"].copy()

# standardize pred vars.
excl_vars = ["clm", "numclaims", "claimcst0", "sample", "id"]
pred_vars = [col for col in df_build.columns if col not in excl_vars]

x_build = df_build[pred_vars].copy()

scaler = StandardScaler()
scaled_x = scaler.fit_transform(x_build)
x_build = pd.DataFrame(
    scaled_x,
    columns=x_build.columns,
    index=x_build.index
)

In [4]:
"""
APPLY VarClusHi.
"""

# apply clustering w/ different lvls of thresholds.
thresholds = [1, 2, 3, 4, 5]
results = []

log.info(f"{'='*80}")
for t in thresholds:
    log.info(f"testing threshold: {t}")
    log.info(f"{'='*80}")
    
    vc = VarClusHi(df=x_build, maxeigval2=t, maxclus=None)
    vc.varclus()
    
    # extract cluster information.
    cluster_info = vc.info
    rs = vc.rsquare
    clusters = sorted(cluster_info["Cluster"].unique())
    n_cluster = cluster_info["Cluster"].nunique()
    cluster_size = cluster_info["Cluster"].value_counts()
    
    # select highest r-squared with own cluster.
    selected_vars= []
    for c in clusters:
        c = int(c)
        cluster_vars = rs[rs["Cluster"] == c]

        # append var w/ highest R^2.
        best_var = cluster_vars.loc[cluster_vars["RS_Own"].idxmax(), "Variable"]
        selected_vars.append(best_var)
    
    # calc. average r-square from selected vars.
    selected_rs = rs[rs["Variable"].isin(selected_vars)]
    avg_rs_within = selected_rs["RS_Own"].mean()
    avg_rs_between = selected_rs["RS_NC"].mean()
    
    # calc. separation.
    sep = avg_rs_within - avg_rs_between
    
    # calc. reduction rate.
    red_rate = (1 - len(selected_vars) / len(pred_vars)) * 100
    
    # store result.
    results.append({
        "threshold": t, 
        "cluster_n": n_cluster,
        "within_rs": avg_rs_within,
        "between_rs":avg_rs_between,
        "seperation": sep,
        "selected_vars": selected_vars
    })
    
    # log.
    log.info("RESULTS:")
    log.info(f"{'-'*80}")
    log.info(f"Clusters: {n_cluster}")
    log.info(f"Selected features: {len(selected_vars)}")
    log.info(f"Reduction rate: {round(red_rate, 2)}%")
    log.info(f"Avg. R2 (within): {round(avg_rs_within, 2)}")
    log.info(f"Avg. R2 (between): {round(avg_rs_between, 2)}")
    log.info(f"Separation: {round(sep, 2)}")
    log.info(f"{'='*80}")
    

# save results to output dir.
result_df = pd.DataFrame.from_records(results)
result_df.to_csv(OUTPUT_DIR / "varreduction_stats.csv")

testing threshold: 1
RESULTS:
--------------------------------------------------------------------------------
Clusters: 27
Selected features: 27
Reduction rate: 49.06%
Avg. R2 (within): 0.75
Avg. R2 (between): 0.02
Separation: 0.72
testing threshold: 2
RESULTS:
--------------------------------------------------------------------------------
Clusters: 3
Selected features: 3
Reduction rate: 94.34%
Avg. R2 (within): 0.81
Avg. R2 (between): 0.0
Separation: 0.81
testing threshold: 3
RESULTS:
--------------------------------------------------------------------------------
Clusters: 1
Selected features: 1
Reduction rate: 98.11%
Avg. R2 (within): 0.45
Avg. R2 (between): 0.0
Separation: 0.45
testing threshold: 4
RESULTS:
--------------------------------------------------------------------------------
Clusters: 1
Selected features: 1
Reduction rate: 98.11%
Avg. R2 (within): 0.45
Avg. R2 (between): 0.0
Separation: 0.45
testing threshold: 5
RESULTS:
-----------------------------------------------